In [ ]:
import pprint

# Use system PAC file to get proxies
from pypac import PACSession
from IPython.display import display, HTML

import pandas as pd

# Use system certificates
# uv add pip-system-certs

In [ ]:
session = PACSession()

SASSTUDIO_HOST = "https://sas8.pf.echonet"
API_URL = f"{SASSTUDIO_HOST}/sasexec"
REMOTE_SESSION_ID = "939a74d4-e309-4f88-8e85-fd81a02b8eb5"
SESSION_URL = f"{API_URL}/sessions/{REMOTE_SESSION_ID}"
LIBRARY_URL = f"{API_URL}/libdata/{REMOTE_SESSION_ID}"


cookies = {
    "35ab575d101a22e5ee8268898c520c61_Cluster2": "961AAB24189BE10D75E37A1986CCC440.71d79d7054eb95eab47728a60d805139_SASServer2_1",
}

headers = {
    "Content-Type": "text/plain; charset=UTF-8",
    "RemoteSession-Id": REMOTE_SESSION_ID,
}

## Run SAS Code

- Submit SAS code for execution
- Print log
- Print result when done
- List output datasets

In [ ]:
params = {
    "label": "Program%201",
    "uri": "Program%201",
    "pdf": "true",
    "rtf": "true",
}

code = "proc sql outobs=10; create table temp as select * from NORDWH.TB_ACCT; select * from dco.loadlog; quit;"

response = session.post(
    url=SESSION_URL + "/asyncSubmissions",
    params=params,
    cookies=cookies,
    headers=headers,
    data=code,
)
response.raise_for_status()

logs = []
while True:
    poll_resp = session.get(
        url=SESSION_URL + "/messages/longpoll",
        # params={
        #     "request.preventCache": str(int(datetime.now().timestamp() * 1000)),
        # },
        cookies=cookies,
        headers=headers,
    )
    poll_res = poll_resp.json()
    print(poll_res)
    logs.extend(
        [
            msg["payload"]["chunk"]
            for msg in poll_res
            if msg["messageType"] in ("LogChunk", "LogEnd")
        ]
    )
    if not poll_res:
        break
    submit_complete_msg = list(
        filter(lambda x: x["messageType"] == "SubmitComplete", poll_res)
    )
    if len(submit_complete_msg) > 0:
        res_link = list(
            filter(
                lambda m: m["rel"] == "results",
                submit_complete_msg[0]["payload"]["links"],
            )
        )[0]["href"]
        res_resp = session.get(
            res_link,
            cookies=cookies,
            headers=headers,
        )
        print(
            "Output Datasets",
            [
                f"{d['library']}.{d['member']}"
                for d in submit_complete_msg[0]["payload"]["dataSets"]
            ],
        )
        display(HTML("\n".join(logs)))
        display(HTML(res_resp.text))

        break


## File Operations

- Get file text content
- Create/update file
- Delete file
- List directory

In [ ]:
remote_file_path = "applis/12201-esmb0-sasbasic/data/DEN/pgm/dev/bin/x_betterEmail2.sh"
file_url = f"{SESSION_URL}/workspace/~~ds~~{remote_file_path}"

In [ ]:
get_file_resp = session.get(
    url=file_url,
    params={'ct': 'text/plain;charset=ISO-8859-1'},
    cookies=cookies,
    headers=headers,
)
get_file_resp.raise_for_status()
print(get_file_resp.text)

In [ ]:
file_text = """
%put hello from ipy;
"""
set_file_resp = session.post(
    url=file_url,
    params={'ct': 'text/plain;charset=ISO-8859-1'},
    cookies=cookies,
    headers=headers,
    data=file_text,
)
set_file_resp.raise_for_status()

In [ ]:
del_file_resp = session.delete(
    url=file_url,
    params={'ct': 'text/plain;charset=ISO-8859-1'},
    cookies=cookies,
    headers=headers,
)
del_file_resp.raise_for_status()

In [ ]:
list_dir_resp = session.get(
    url=f"{API_URL}/{REMOTE_SESSION_ID}" + "/_root_",
    cookies=cookies,
    headers=headers,
)
list_dir_resp.raise_for_status()
pprint.pp(list_dir_resp.json())

## Library/Datasets Operations

- List libraries
- List table columns
- Get table data

In [ ]:
list_lib_resp = session.get(
    url=f"{LIBRARY_URL}/libraries",
    # url=f"{LIBRARY_URL}/libraries~SASHELP",
    cookies=cookies,
    headers=headers,
)
list_lib_resp.raise_for_status()
pprint.pp(list_lib_resp.json())

In [ ]:
library, table = "SASHELP.AIR".split(".")
get_tbl_col_resp = session.post(
    url=f"{SESSION_URL}/tables/{library}/{table}/",
    params={"getViewColumnCount": "true"},
    cookies=cookies,
    headers={
        **headers,
        "Content-Type": "application/json",
    },
    data={
        "isMultipleWorkspace": "",
        "serverName": "",
        "dataSetKey": "",
        "clearCache": "false",
    },
)
get_tbl_col_resp.raise_for_status()
pprint.pp(get_tbl_col_resp.json())

In [ ]:
library, table = "SASHELP.AIR".split(".")
get_tbl_data_resp = session.post(
    url=f"{SESSION_URL}/sql",
    params={"numobs": 101},
    cookies=cookies,
    headers={
        **headers,
        "Content-Type": "application/json",
    },
    data=f"select * from {library}.'{table}'n(firstobs=1 obs=102)",
)
get_tbl_data_resp.raise_for_status()
# pprint.pp(get_tbl_data_resp.json())
data=get_tbl_data_resp.json()
df = pd.DataFrame(data['rows'], columns=data['colNames'])
df